# Part 1–2: Team Schedule Summary from games.csv

## Problem Statement  

Using the file games.csv, we want to build a clear **per-team summary** of the schedule.  
For each team we must compute:

- **(a) Home dates** – all dates on which the team plays at home, ordered chronologically  
- **(b) Home opponent counts** – for every opponent, how many times they visit this team  
- **(c) Away opponent counts** – for every opponent, how many times this team visits them  
- **(d) Away dates** – all dates on which the team plays away, ordered chronologically  

The goal of Parts 1 and 2 is to load the data, compute these statistics, and print a human-readable summary for each team.

---

## Approach  

We use **Python + pandas** to read the CSV, clean the columns, and aggregate the information by team.  
The code is structured into three main pieces:

1. **Data loading and cleaning – load_data**
2. **Computation of team-level statistics – compute_team_info**
3. **Pretty printing of the results – print_summary**

At the end, a short “main script” calls these functions to produce the full report.

---

## Step 1: Load data from CSV  

We first mount Google Drive and load the file:

- Mounts the drive with  
  drive.mount("/content/drive")
- Sets the CSV path, e.g.  
  CSV_PATH = "/content/drive/MyDrive/games.csv"`
- Reads the file into a pandas DataFrame with pd.read_csv.

Inside **load_data(csv_path)**:

- We **standardize column names** so the rest of the code can always rely on the columns  
  Date, Home, and Away (renaming Visitor → Away if needed).
- We create a new column **Date_parsed** using pd.to_datetime to convert the date strings into real datetime objects.  
  This allows us to sort games chronologically.

The function returns the cleaned DataFrame df.

---

## Step 2: Compute team-level information  

The function **compute_team_info(df)** builds a dictionary with all the required information for each team:

- First we build the set of all teams  
  teams = sorted(set(df["Home"]).union(df["Away"])).
- For each team t:
  - We select all **home games** of t and sort them by Date_parsed
    → list of **home dates** for (a).
  - We count, using value_counts, how many times each opponent appears as **Away** in those home games  
    → **home opponent counts** for (b).
  - We select all **away games** of t (where t appears as `Away`) and sort them  
    → list of **away dates** for (d).
  - We count how many times each opponent appears as **Home** in those away games  
    → **away opponent counts** for (c).

For each team we store an OrderedDict with four entries:

- "Home Dates"
- "Home Opponents Count"
- "Away Opponents Count"  
- "Away Dates"

The function returns a dictionary results indexed by team name.

---

## Step 3: Print the summary  

The function **`print_summary(results, max_width=100)`** prints a clean report for every team:

- Draws separator lines (= and -) to visually separate teams.
- For each team, prints:

  1. **(a) Home Dates** – one date per line  
  2. **(b) Times played at HOME vs each opponent** – a bullet list “vs Opponent: count”  
  3. **(c) Times played AWAY vs each opponent** – a bullet list “at Opponent: count”  
  4. **(d) Away Dates** – one date per line  

- If any list is empty, the function prints (none) to make it explicit.

This produces the long formatted output shown in the notebook, with one clearly separated block per team.

---

## Step 4: Main script  

Finally, the bottom of the cell runs the full pipeline:

1. df = load_data(CSV_PATH)  
   – loads and cleans the schedule from games.csv.
2. results = compute_team_info(df)  
   – computes all home/away information for every team.
3. print_summary(results)
   – prints the full per-team summary used for answering Parts 1 and 2.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Data download and formatting
import pandas as pd
import json
from collections import OrderedDict


# >>> CHANGE THIS PATH IF NEEDED <<<
# If games.csv is inside the project folder use, for example:
# CSV_PATH = "/content/drive/MyDrive/Project 2 - Carolina Cougars/Material/games.csv"
CSV_PATH = "/content/drive/MyDrive/games.csv"

def load_data(csv_path: str) -> pd.DataFrame:
    """Load CSV and make sure column names are consistent."""
    df = pd.read_csv(csv_path)

    # Rename columns if necessary so we always have: Date, Home, Away
    rename_map = {}
    if "Visitor" in df.columns:
        rename_map["Visitor"] = "Away"
    if "Home" in df.columns:
        rename_map["Home"] = "Home"
    if "Date" in df.columns:
        rename_map["Date"] = "Date"

    df = df.rename(columns=rename_map)

    # Parse dates to sort them later
    df["Date_parsed"] = pd.to_datetime(df["Date"], errors="coerce")
    return df

def compute_team_info(df: pd.DataFrame) -> dict:
    """Compute home/away dates and opponent counts for each team."""
    teams = sorted(set(df["Home"]).union(df["Away"]))
    results = {}

    for team in teams:
        home_matches = df[df["Home"] == team].copy()
        away_matches = df[df["Away"] == team].copy()

        # (a) home dates (sorted if possible)
        home_dates = home_matches.sort_values("Date_parsed")["Date"].tolist()
        # (b) home counts by opponent
        home_vs_counts = home_matches["Away"].value_counts().sort_index().to_dict()
        # (c) away counts by opponent
        away_vs_counts = away_matches["Home"].value_counts().sort_index().to_dict()
        # (d) away dates (sorted if possible)
        away_dates = away_matches.sort_values("Date_parsed")["Date"].tolist()

        results[team] = OrderedDict([
            ("Home Dates", home_dates),
            ("Home Opponents Count", {k: int(v) for k, v in home_vs_counts.items()}),
            ("Away Opponents Count", {k: int(v) for k, v in away_vs_counts.items()}),
            ("Away Dates", away_dates),
        ])

    return results

def print_summary(results: dict, max_width: int = 100):
    """Pretty-print the summary for each team."""
    line = "=" * max_width
    sep = "-" * max_width

    for team in sorted(results.keys()):
        info = results[team]
        print(line)
        print(f"Team: {team}")
        print(sep)

        # (a)
        print("(a) Home Dates:")
        if info["Home Dates"]:
            for d in info["Home Dates"]:
                print(f"   • {d}")
        else:
            print("   (none)")

        # (b)
        print("\n(b) Times played at HOME vs each opponent:")
        if info["Home Opponents Count"]:
            for opp, cnt in info["Home Opponents Count"].items():
                print(f"   • vs {opp}: {cnt}")
        else:
            print("   (none)")

        # (c)
        print("\n(c) Times played AWAY vs each opponent:")
        if info["Away Opponents Count"]:
            for opp, cnt in info["Away Opponents Count"].items():
                print(f"   • at {opp}: {cnt}")
        else:
            print("   (none)")

        # (d)
        print("\n(d) Away Dates:")
        if info["Away Dates"]:
            for d in info["Away Dates"]:
                print(f"   • {d}")
        else:
            print("   (none)")
        print()


# Load data
df = load_data(CSV_PATH)

# Compute results dictionary
results = compute_team_info(df)

# Print summary
print_summary(results)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Team: Atlanta Hawks
----------------------------------------------------------------------------------------------------
(a) Home Dates:
   • Mon, Nov 03, 2025
   • Fri, Nov 07, 2025
   • Sat, Nov 15, 2025
   • Mon, Nov 17, 2025
   • Wed, Nov 19, 2025
   • Sun, Nov 23, 2025
   • Thu, Nov 27, 2025
   • Fri, Nov 28, 2025
   • Sat, Nov 29, 2025
   • Thu, Dec 25, 2025

(b) Times played at HOME vs each opponent:
   • vs Boston Celtics: 1
   • vs Chicago Bulls: 1
   • vs Dallas Mavericks: 1
   • vs Golden State Warriors: 1
   • vs Los Angeles Lakers: 1
   • vs Miami Heat: 1
   • vs Milwaukee Bucks: 1
   • vs New York Knicks: 1
   • vs Phoenix Suns: 1
   • vs Toronto Raptors: 1

(c) Times played AWAY vs each opponent:
   • at Brooklyn Nets: 1
   • at Chicago Bulls: 1
   • at Cleveland Cavaliers: 1
   • at Denver Nuggets: 1
   • at Houston Rockets: 1
   • at Philadel

# Part 3: NBA Schedule with Time Zone Constraints

## Problem Statement
Compute a feasible schedule that satisfies the following constraint: **no team should play three consecutive matches where the sum of the absolute values of the difference between the time zones of two consecutive matches is 4 or more**.

For example, if a team plays games 1, 2, 3 and:
- The time zone difference between game 1 and game 2 is 2
- The time zone difference between game 2 and game 3 is 3
Then the schedule is infeasible, since |2| + |3| = 5 ≥ 4.

## Approach
We'll use Integer Programming (Gurobi) to find a feasible schedule that:
1. Maintains all constraints from Part 2 (same home/away dates, same number of games between teams)
2. Adds the time zone constraint for consecutive games

---

## Step 1: Install Required Libraries


### Install pytz library
Installs the `pytz` library which is needed to handle timezone conversions and calculate UTC offsets for different arenas.


In [ ]:
!pip install pytz

### Load Data
- Imports necessary libraries: `pandas` for data manipulation, `datetime` for date handling, and `pytz` for timezone operations
- Loads the games schedule from `games.csv`
- Displays the first few rows to verify the data loaded correctly


### Extract Unique Arenas
Extracts all unique arena names from the games data. This helps us identify which arenas need timezone mappings.


### Map Arenas to Timezones
Creates a dictionary mapping each NBA arena to its corresponding timezone name (using IANA timezone identifiers). This is necessary because:
- Each arena is in a different city with a specific timezone
- We need to calculate UTC offsets to measure timezone differences between consecutive games
- Examples: TD Garden (Boston) is in Eastern Time, Crypto.com Arena (LA) is in Pacific Time


### Check for Missing Arenas
Verifies that all arenas in the data have been mapped to timezones. If any arenas are missing, this will identify them so we can add their timezone mappings.


### Function to Calculate UTC Offset
Defines a helper function `tz_offset_hours()` that:
- Takes a timezone name and an optional reference datetime
- Returns the UTC offset in hours as an integer (e.g., -5 for Eastern Time, -8 for Pacific Time)
- Uses a reference datetime to handle Daylight Saving Time (DST) consistently
- The offset is negative because US timezones are behind UTC


### Calculate UTC Offsets for All Arenas
- Sets a reference date (November 1, 2025) to ensure consistent DST handling throughout the season
- Creates a dictionary mapping each arena to its UTC offset in hours
- This offset will be used in the optimization model to calculate timezone differences between consecutive games
- Example outputs: TD Garden → -4 (Eastern), Crypto.com Arena → -7 (Pacific)


### Add Timezone Information to Games DataFrame
- Adds two new columns to the games DataFrame:
  - `ArenaTZName`: The timezone name for each arena
  - `ArenaUTCOffset`: The UTC offset in hours for each arena
- This makes it easy to look up timezone information for each game in the optimization model


---

## Step 2: Set Up the Integer Programming Model

### Import Optimization Libraries
Imports Gurobi optimization library (`gurobipy`) which will be used to solve the integer programming problem. Gurobi is a powerful commercial solver for linear and integer programming.


### Define Sets and Parameters
This cell sets up all the data structures needed for the optimization model:

- **G**: List of game indices (0 to 127, representing all 128 games)
- **T**: List of all teams (16 teams in the NBA)
- **D**: List of date indices (0 to R-1, where R is the number of unique dates)
- **date_to_idx**: Dictionary mapping actual date strings to numeric indices
- **tz_game**: Dictionary mapping each game to its arena's UTC offset
- **home/away**: Dictionaries mapping each game to its home and away teams

**IMPORTANT**: Dates are sorted **chronologically** (not alphabetically) so that DateSlot indices correspond to chronological order. This ensures that checking consecutive DateSlots (d, d+1, d+2) actually checks chronologically consecutive games, which is required for the timezone constraint to work correctly.

These will be used as indices for the decision variables in the optimization model.


### Create Model and Decision Variables
Creates the Gurobi optimization model and defines all decision variables:

1. **x[g,d]**: Binary variable = 1 if game g is scheduled on day d, 0 otherwise
   - This is the main decision variable that determines the schedule

2. **plays[t,d]**: Binary variable = 1 if team t plays on day d, 0 otherwise
   - Helper variable to track when each team plays

3. **tz_var[t,d]**: Continuous variable representing the timezone (UTC offset) of team t's game on day d
   - Used to track the timezone location of each team on each day

4. **a1[t,d], a2[t,d]**: Continuous variables ≥ 0 representing absolute timezone differences
   - a1[t,d] = |tz[t,d+1] - tz[t,d]| (difference between day d and d+1)
   - a2[t,d] = |tz[t,d+2] - tz[t,d+1]| (difference between day d+1 and d+2)

5. **diff1[t,d], diff2[t,d]**: Continuous variables representing signed timezone differences
   - diff1[t,d] = tz[t,d+1] - tz[t,d]
   - diff2[t,d] = tz[t,d+2] - tz[t,d+1]
   - Used to compute the absolute values a1 and a2


### Constraint (a) - Each Game Scheduled Exactly Once
Ensures that every game is scheduled on exactly one day. For each game g, the sum of x[g,d] over all days d must equal 1.

**Mathematical formulation**: ∑<sub>d∈D</sub> x[g,d] = 1 for all g ∈ G

This guarantees that all 128 games appear in the final schedule.


### Constraint (b) - Each Team Plays At Most One Game Per Day
Ensures that no team plays more than one game on any given day. For each team t and day d, the sum of all games involving team t on day d must be ≤ 1.

**Mathematical formulation**: ∑<sub>g: t plays in g</sub> x[g,d] ≤ 1 for all t ∈ T, d ∈ D

This prevents scheduling conflicts where a team would need to be in two places at once.


### Constraint (c) - Define Helper Variables
Defines two important helper variables:

1. **plays[t,d]**: Set equal to 1 if team t plays any game on day d
   - plays[t,d] = ∑<sub>g: t plays in g</sub> x[g,d]
   - This tracks whether a team is active on a given day

2. **tz_var[t,d]**: Set equal to the timezone offset of team t's game on day d
   - tz_var[t,d] = ∑<sub>g: t plays in g</sub> tz_game[g] × x[g,d]
   - If team t plays game g on day d, then tz_var[t,d] = tz_game[g] (the arena's UTC offset)
   - If team t doesn't play on day d, tz_var[t,d] = 0

These variables are essential for enforcing the timezone constraint.


### The Timezone Constraint (Core of Part 3)
This is the most important constraint for Part 3. It enforces that **no team plays three consecutive games where the sum of absolute timezone differences is ≥ 4**.

**The constraint logic:**
For each team t and each triple of consecutive days (d, d+1, d+2):

1. **Calculate differences:**
   - diff1[t,d] = tz_var[t,d+1] - tz_var[t,d] (timezone jump from day d to d+1)
   - diff2[t,d] = tz_var[t,d+2] - tz_var[t,d+1] (timezone jump from day d+1 to d+2)

2. **Calculate absolute values:**
   - a1[t,d] ≥ diff1[t,d] and a1[t,d] ≥ -diff1[t,d] → a1[t,d] = |diff1[t,d]|
   - a2[t,d] ≥ diff2[t,d] and a2[t,d] ≥ -diff2[t,d] → a2[t,d] = |diff2[t,d]|

3. **Enforce the constraint:**
   - If team t plays on all three days (d, d+1, d+2), then: a1[t,d] + a2[t,d] ≤ 3
   - If team t doesn't play on all three days, the constraint is relaxed using Big-M method
   - The Big-M constraint: a1[t,d] + a2[t,d] ≤ 3 + M × (3 - (plays[t,d] + plays[t,d+1] + plays[t,d+2]))
   - When the team plays all three days: plays[t,d] + plays[t,d+1] + plays[t,d+2] = 3, so the constraint becomes a1 + a2 ≤ 3
   - When the team doesn't play all three days: the right-hand side becomes ≥ 3 + M, effectively disabling the constraint

**Example:** If a team plays in timezone -4 (Eastern) on day d, -7 (Pacific) on day d+1, and -4 (Eastern) on day d+2:
- diff1 = -7 - (-4) = -3, so a1 = 3
- diff2 = -4 - (-7) = 3, so a2 = 3
- a1 + a2 = 6 ≥ 4, so this violates the constraint and won't be allowed


### Solve the Model
- Sets a dummy objective function (minimize 0) since we only need to find a feasible solution, not an optimal one
- Enables solver output to see the optimization progress
- Calls `m.optimize()` to solve the integer programming problem
- Prints the model status (2 = OPTIMAL, 3 = INFEASIBLE, etc.)

The solver will search for a schedule that satisfies all constraints. If a feasible solution exists, it will be found. If not, the solver will report that the problem is infeasible.


### Cell 13b: Part 2 Constraints - Preserve Original Home/Away Dates

This cell enforces Part 2 constraints (e) and (f):

- **Constraint (e)**: Each team must play home on the same dates as the original schedule
- **Constraint (f)**: Each team must play away on the same dates as the original schedule

**Mathematical formulation:**

For constraint (e) - Home dates:
- For each team t and each original home date d: ∑<sub>g: home[g] = t</sub> x[g, date_to_idx[d]] = 1
- This ensures team t plays exactly one home game on date d

For constraint (f) - Away dates:
- For each team t and each original away date d: ∑<sub>g: away[g] = t</sub> x[g, date_to_idx[d]] = 1
- This ensures team t plays exactly one away game on date d

These constraints preserve the original schedule's home/away date structure while allowing games to be rescheduled to different dates (subject to the timezone constraint).


### Extract the Solution
After the solver finds a feasible solution, this cell:
- Creates a reverse mapping from date indices back to actual date strings
- Iterates through all games and days to find which games are scheduled on which days (where x[g,d].X > 0.5)
- Builds a schedule DataFrame with game information, including visitor, home team, and scheduled date
- Sorts the schedule by date slot and game index for easy reading


### Export Final Schedule to CSV
Creates a comprehensive schedule DataFrame with all relevant information:
- Game index, date slot, and actual date string
- Visitor and home teams
- Arena name and UTC offset

Then exports the schedule to `final_schedule.csv` for easy sharing and verification. This file can be used to verify that the timezone constraints are satisfied and to compare with the original schedule.


### Cell 1: Load Data
- Imports necessary libraries: `pandas` for data manipulation, `datetime` for date handling, and `pytz` for timezone operations
- Loads the games schedule from `games.csv`
- Displays the first few rows to verify the data loaded correctly


In [ ]:
import pandas as pd
from datetime import datetime
import pytz

CSV_PATH = "/content/drive/MyDrive/games.csv"
games = pd.read_csv(CSV_PATH)

# Quick peek
games.head()

,Date,Visitor,PTS,Home,PTS.1,Attend.,LOG,Arena,Notes
0,"Sat, Nov 01, 2025",Golden State Warriors,NaN,Boston Celtics,NaN,"19,000",7:30 PM,TD Garden,NaN
1,"Sat, Nov 01, 2025",Los Angeles Lakers,NaN,New York Knicks,NaN,"19,400",7:30 PM,Madison Square Garden,NaN
2,"Sat, Nov 01, 2025",Denver Nuggets,NaN,Brooklyn Nets,NaN,"17,500",7:30 PM,Barclays Center,NaN
3,"Sat, Nov 01, 2025",Phoenix Suns,NaN,Philadelphia 76ers,NaN,"19,650",7:30 PM,Wells Fargo Center,NaN
4,"Sat, Nov 01, 2025",Houston Rockets,NaN,Toronto Raptors,NaN,"19,600",7:30 PM,Scotiabank Arena,NaN


### Cell 2: Extract Unique Arenas
Extracts all unique arena names from the games data. This helps us identify which arenas need timezone mappings.


In [ ]:
arenas = games["Arena"].unique()
arenas

array(['TD Garden', 'Madison Square Garden', 'Barclays Center',
       'Wells Fargo Center', 'Scotiabank Arena', 'Kaseya Center',
       'United Center', 'Fiserv Forum', 'Crypto.com Arena',
       'Chase Center', 'Footprint Center', 'Toyota Center',
       'American Airlines Center', 'State Farm Arena',
       'Rocket Mortgage FieldHouse', 'Ball Arena'], dtype=object)

### Cell 3: Map Arenas to Timezones
Creates a dictionary mapping each NBA arena to its corresponding timezone name (using IANA timezone identifiers). This is necessary because:
- Each arena is in a different city with a specific timezone
- We need to calculate UTC offsets to measure timezone differences between consecutive games
- Examples: TD Garden (Boston) is in Eastern Time, Crypto.com Arena (LA) is in Pacific Time


In [ ]:
arena_timezone = {
    "TD Garden": "America/New_York",                # Boston
    "Madison Square Garden": "America/New_York",    # New York
    "Barclays Center": "America/New_York",          # Brooklyn
    "Wells Fargo Center": "America/New_York",       # Philadelphia
    "Scotiabank Arena": "America/Toronto",          # Toronto
    "Kaseya Center": "America/New_York",            # Miami
    "United Center": "America/Chicago",             # Chicago
    "Fiserv Forum": "America/Chicago",              # Milwaukee
    "Crypto.com Arena": "America/Los_Angeles",      # Los Angeles
    "Chase Center": "America/Los_Angeles",          # San Francisco
    "Footprint Center": "America/Phoenix",          # Phoenix
    "Toyota Center": "America/Chicago",             # Houston
    "American Airlines Center": "America/Chicago",  # Dallas
    "State Farm Arena": "America/New_York",         # Atlanta
    "Rocket Mortgage FieldHouse": "America/New_York", # Cleveland
    "Ball Arena": "America/Denver"                  # Denver
}

### Cell 4: Check for Missing Arenas
Verifies that all arenas in the data have been mapped to timezones. If any arenas are missing, this will identify them so we can add their timezone mappings.


In [ ]:
missing_arenas = [a for a in arenas if a not in arena_timezone]
missing_arenas

[]

### Cell 5: Function to Calculate UTC Offset
Defines a helper function `tz_offset_hours()` that:
- Takes a timezone name and an optional reference datetime
- Returns the UTC offset in hours as an integer (e.g., -5 for Eastern Time, -8 for Pacific Time)
- Uses a reference datetime to handle Daylight Saving Time (DST) consistently
- The offset is negative because US timezones are behind UTC


In [ ]:
def tz_offset_hours(tz_name, ref_datetime=None):
    """
    Return integer UTC offset (in hours) for a given timezone.
    ref_datetime lets you fix a date (for DST issues).
    """
    if ref_datetime is None:
        ref_datetime = datetime.now()
    tz = pytz.timezone(tz_name)
    offset = tz.utcoffset(ref_datetime).total_seconds() / 3600
    return int(offset)

### Cell 6: Calculate UTC Offsets for Each Game Based on Actual Date
- **Calculates timezone offset for each game individually** based on the game's actual date
- This ensures **100% accurate DST handling** - games on Nov 1-2 use DST offsets, games on Nov 3+ use standard time offsets
- DST ends on November 2, 2025 (first Sunday), so:
  - **Nov 1-2 games**: Use DST offsets (EDT=-4, CDT=-5, MDT=-6, PDT=-7)
  - **Nov 3+ games**: Use standard time offsets (EST=-5, CST=-6, MST=-7, PST=-8)
  - **Arizona (Phoenix)**: Always -7 (doesn't observe DST)
- This offset will be used in the optimization model to calculate timezone differences between consecutive games
- **No approximation** - every game has the correct offset for its specific date


In [ ]:
# Calculate timezone offset for each game based on its actual date
# This ensures correct DST handling - each game uses the offset for its specific date
# DST ends on Nov 2, 2025, so games on Nov 1-2 use DST offsets, Nov 3+ use standard time

def parse_date_string(date_str):
    """Parse date string in format 'Day, Mon DD, YYYY' to datetime object"""
    date_str = date_str.strip('"')
    try:
        return datetime.strptime(date_str, "%a, %b %d, %Y")
    except:
        return None

# Calculate offset for each game based on its date
# This handles DST correctly - Nov 1-2 games get DST offsets, Nov 3+ get standard time
games["ArenaUTCOffset"] = games.apply(
    lambda row: tz_offset_hours(
        arena_timezone[row["Arena"]],
        parse_date_string(row["Date"])
    ) if parse_date_string(row["Date"]) else None,
    axis=1
)

# Verify offsets are calculated correctly
print("Timezone offsets calculated per game based on actual date:")
print("Sample of first 10 games:")
print(games[["Date", "Arena", "ArenaUTCOffset"]].head(10))
print("\nOffset summary by date (to verify DST handling):")
print(games.groupby("Date")["ArenaUTCOffset"].value_counts().head(20))

Timezone offsets calculated per game based on actual date:
Sample of first 10 games:
                Date                  Arena  ArenaUTCOffset
0  Sat, Nov 01, 2025              TD Garden              -4
1  Sat, Nov 01, 2025  Madison Square Garden              -4
2  Sat, Nov 01, 2025        Barclays Center              -4
3  Sat, Nov 01, 2025     Wells Fargo Center              -4
4  Sat, Nov 01, 2025       Scotiabank Arena              -4
5  Sat, Nov 01, 2025          Kaseya Center              -4
6  Sat, Nov 01, 2025          United Center              -5
7  Sat, Nov 01, 2025           Fiserv Forum              -5
8  Mon, Nov 03, 2025       Crypto.com Arena              -8
9  Mon, Nov 03, 2025           Chase Center              -8

Offset summary by date (to verify DST handling):
Date               ArenaUTCOffset
Fri, Nov 07, 2025  -5                5
                   -6                2
                   -7                1
Fri, Nov 21, 2025  -6                3
               

### Cell 7: Add Timezone Information to Games DataFrame
- Adds two new columns to the games DataFrame:
  - `ArenaTZName`: The timezone name for each arena
  - `ArenaUTCOffset`: The UTC offset in hours for each arena
- This makes it easy to look up timezone information for each game in the optimization model


In [ ]:
# Add timezone name column (already have ArenaUTCOffset from previous cell)
games["ArenaTZName"] = games["Arena"].map(arena_timezone)

# Verify the data
games.head(20)

,Date,Visitor,PTS,Home,PTS.1,Attend.,LOG,Arena,Notes,ArenaUTCOffset,ArenaTZName
0,"Sat, Nov 01, 2025",Golden State Warriors,NaN,Boston Celtics,NaN,"19,000",7:30 PM,TD Garden,NaN,-4,America/New_York
1,"Sat, Nov 01, 2025",Los Angeles Lakers,NaN,New York Knicks,NaN,"19,400",7:30 PM,Madison Square Garden,NaN,-4,America/New_York
2,"Sat, Nov 01, 2025",Denver Nuggets,NaN,Brooklyn Nets,NaN,"17,500",7:30 PM,Barclays Center,NaN,-4,America/New_York
3,"Sat, Nov 01, 2025",Phoenix Suns,NaN,Philadelphia 76ers,NaN,"19,650",7:30 PM,Wells Fargo Center,NaN,-4,America/New_York
4,"Sat, Nov 01, 2025",Houston Rockets,NaN,Toronto Raptors,NaN,"19,600",7:30 PM,Scotiabank Arena,NaN,-4,America/Toronto
5,"Sat, Nov 01, 2025",Dallas Mavericks,NaN,Miami Heat,NaN,"19,700",7:30 PM,Kaseya Center,NaN,-4,America/New_York
6,"Sat, Nov 01, 2025",Atlanta Hawks,NaN,Chicago Bulls,NaN,"20,850",7:30 PM,United Center,NaN,-5,America/Chicago
7,"Sat, Nov 01, 2025",Cleveland Cavaliers,NaN,Milwaukee Bucks,NaN,"17,550",7:30 PM,Fiserv Forum,NaN,-5,America/Chicago
8,"Mon, Nov 03, 2025",Boston Celtics,NaN,Los Angeles Lakers,NaN,"19,000",7:30 PM,Crypto.com Arena,NaN,-8,America/Los_Angeles
9,"Mon, Nov 03, 2025",Denver Nuggets,NaN,Golden State Warriors,NaN,"18,500",7:30 PM,Chase Center,NaN,-8,America/Los_Angeles


---

## Step 2: Set Up the Integer Programming Model

### Cell 8: Import Optimization Libraries
Imports Gurobi optimization library (`gurobipy`) which will be used to solve the integer programming problem. Gurobi is a powerful commercial solver for linear and integer programming.


In [ ]:
!pip install gurobipy
import pandas as pd
from gurobipy import Model, GRB, quicksum


# Quick sanity check
games.head()

Traceback (most recent call last):
  File "/usr/local/bin/pip3", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/main.py", line 78, in main
    command = create_command(cmd_name, isolated=("--isolated" in cmd_args))
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/__init__.py", line 114, in create_command
    module = importlib.import_module(module_path)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unloc

,Date,Visitor,PTS,Home,PTS.1,Attend.,LOG,Arena,Notes,ArenaUTCOffset,ArenaTZName
0,"Sat, Nov 01, 2025",Golden State Warriors,NaN,Boston Celtics,NaN,"19,000",7:30 PM,TD Garden,NaN,-4,America/New_York
1,"Sat, Nov 01, 2025",Los Angeles Lakers,NaN,New York Knicks,NaN,"19,400",7:30 PM,Madison Square Garden,NaN,-4,America/New_York
2,"Sat, Nov 01, 2025",Denver Nuggets,NaN,Brooklyn Nets,NaN,"17,500",7:30 PM,Barclays Center,NaN,-4,America/New_York
3,"Sat, Nov 01, 2025",Phoenix Suns,NaN,Philadelphia 76ers,NaN,"19,650",7:30 PM,Wells Fargo Center,NaN,-4,America/New_York
4,"Sat, Nov 01, 2025",Houston Rockets,NaN,Toronto Raptors,NaN,"19,600",7:30 PM,Scotiabank Arena,NaN,-4,America/Toronto


### Cell 9: Define Sets and Parameters
This cell sets up all the data structures needed for the optimization model:

- **G**: List of game indices (0 to 127, representing all 128 games)
- **T**: List of all teams (16 teams in the NBA)
- **D**: List of date indices (0 to R-1, where R is the number of unique dates)
- **date_to_idx**: Dictionary mapping actual date strings to numeric indices
- **tz_game**: Dictionary mapping each game to its arena's UTC offset
- **home/away**: Dictionaries mapping each game to its home and away teams

**CRITICAL**: Dates are sorted **chronologically** (not alphabetically) so that DateSlot indices correspond to chronological order. This ensures that checking consecutive DateSlots (d, d+1, d+2) actually checks chronologically consecutive games, which is required for the timezone constraint to work correctly.

These will be used as indices for the decision variables in the optimization model.


In [ ]:
# Game indices
G = list(games.index)  # [0, 1, 2, ..., 127] for 128 games
G

# Teams
teams = sorted(set(games["Home"]).union(games["Visitor"]))
T = teams

# Dates / rounds (we'll treat each unique Date as a time slot index)
# IMPORTANT: Sort dates CHRONOLOGICALLY, not alphabetically
# This ensures DateSlot indices correspond to chronological order

def parse_date_for_sorting(date_str):
    """Parse date string to datetime object for chronological sorting"""
    date_str = date_str.strip('"')
    try:
        return datetime.strptime(date_str, "%a, %b %d, %Y")
    except:
        return None

# Get unique dates and sort them chronologically (not alphabetically)
unique_dates = games["Date"].unique()
dates = sorted(unique_dates, key=lambda d: parse_date_for_sorting(d) if parse_date_for_sorting(d) else datetime.max)
D = list(range(len(dates)))  # 0..R-1

# Map actual date string -> index (now indices are in chronological order)
date_to_idx = {d: i for i, d in enumerate(dates)}

# Verify chronological order
print("Date indices in chronological order:")
for i, d in enumerate(dates):
    print(f"  DateSlot {i}: {d}")

# Time-zone offset for each game g when scheduled on date d
# CRITICAL: Calculate offsets based on SCHEDULED date, not original date
# This ensures DST is handled correctly when games are rescheduled
tz_game_date = {}
for g in G:
    arena = games.loc[g, "Arena"]
    arena_tz = arena_timezone[arena]
    for d in D:
        # Get the date for this DateSlot
        scheduled_date_str = dates[d]
        scheduled_date = parse_date_for_sorting(scheduled_date_str)
        if scheduled_date:
            # Calculate offset for this specific date (handles DST correctly)
            tz_game_date[g, d] = tz_offset_hours(arena_tz, scheduled_date)
        else:
            # Fallback: use offset from original date
            tz_game_date[g, d] = int(games.loc[g, "ArenaUTCOffset"])

print(f"Calculated timezone offsets for {len(G)} games × {len(D)} dates = {len(tz_game_date)} combinations")
print("Sample offsets (Game 0 on different dates):")
for d in D[:3]:
    print(f"  DateSlot {d} ({dates[d]}): {tz_game_date[0, d]}")

# Helper: which teams play in each game
home = {g: games.loc[g, "Home"] for g in G}
away = {g: games.loc[g, "Visitor"] for g in G}

Date indices in chronological order:
  DateSlot 0: Sat, Nov 01, 2025
  DateSlot 1: Mon, Nov 03, 2025
  DateSlot 2: Wed, Nov 05, 2025
  DateSlot 3: Fri, Nov 07, 2025
  DateSlot 4: Tue, Nov 11, 2025
  DateSlot 5: Thu, Nov 13, 2025
  DateSlot 6: Sat, Nov 15, 2025
  DateSlot 7: Mon, Nov 17, 2025
  DateSlot 8: Wed, Nov 19, 2025
  DateSlot 9: Fri, Nov 21, 2025
  DateSlot 10: Sun, Nov 23, 2025
  DateSlot 11: Thu, Nov 27, 2025
  DateSlot 12: Fri, Nov 28, 2025
  DateSlot 13: Sat, Nov 29, 2025
  DateSlot 14: Mon, Dec 01, 2025
  DateSlot 15: Thu, Dec 25, 2025
Calculated timezone offsets for 128 games × 16 dates = 2048 combinations
Sample offsets (Game 0 on different dates):
  DateSlot 0 (Sat, Nov 01, 2025): -4
  DateSlot 1 (Mon, Nov 03, 2025): -5
  DateSlot 2 (Wed, Nov 05, 2025): -5


### Cell 10: Create Model and Decision Variables
Creates the Gurobi optimization model and defines all decision variables:

1. **x[g,d]**: Binary variable = 1 if game g is scheduled on day d, 0 otherwise
   - This is the main decision variable that determines the schedule

2. **plays[t,d]**: Binary variable = 1 if team t plays on day d, 0 otherwise
   - Helper variable to track when each team plays

3. **tz_var[t,d]**: Continuous variable representing the timezone (UTC offset) of team t's game on day d
   - Used to track the timezone location of each team on each day

4. **a1[t,d], a2[t,d]**: Continuous variables ≥ 0 representing absolute timezone differences
   - a1[t,d] = |tz[t,d+1] - tz[t,d]| (difference between day d and d+1)
   - a2[t,d] = |tz[t,d+2] - tz[t,d+1]| (difference between day d+1 and d+2)

5. **diff1[t,d], diff2[t,d]**: Continuous variables representing signed timezone differences
   - diff1[t,d] = tz[t,d+1] - tz[t,d]
   - diff2[t,d] = tz[t,d+2] - tz[t,d+1]
   - Used to compute the absolute values a1 and a2


In [ ]:
m = Model("NBA_TimeZone_Gurobi")

# x[g,d]: 1 if game g is scheduled on day d
x = m.addVars(G, D, vtype=GRB.BINARY, name="x")

# plays[t,d]: 1 if team t plays on day d
plays = m.addVars(T, D, vtype=GRB.BINARY, name="plays")

# tz_var[t,d]: time zone of team t's game on day d
# offsets in your data are around -4..-7, but we allow a safe range [-12, 12]
tz_var = m.addVars(T, D, lb=-12, ub=12, vtype=GRB.CONTINUOUS, name="tz")

# Absolute-diff helper vars for consecutive games
a1 = m.addVars(T, D, lb=0.0, vtype=GRB.CONTINUOUS, name="a1")
a2 = m.addVars(T, D, lb=0.0, vtype=GRB.CONTINUOUS, name="a2")

# Signed difference vars (tz_{d+1} - tz_d and tz_{d+2} - tz_{d+1})
diff1 = m.addVars(T, D, lb=-24, ub=24, vtype=GRB.CONTINUOUS, name="diff1")
diff2 = m.addVars(T, D, lb=-24, ub=24, vtype=GRB.CONTINUOUS, name="diff2")

### Cell 11: Constraint (a) - Each Game Scheduled Exactly Once
Ensures that every game is scheduled on exactly one day. For each game g, the sum of x[g,d] over all days d must equal 1.

**Mathematical formulation**: ∑<sub>d∈D</sub> x[g,d] = 1 for all g ∈ G

This guarantees that all 128 games appear in the final schedule.


In [ ]:
#(a) Each game is scheduled exactly once
for g in G:
    m.addConstr(quicksum(x[g, d] for d in D) == 1, name=f"one_day_per_game_{g}")

### Cell 12: Constraint (b) - Each Team Plays At Most One Game Per Day
Ensures that no team plays more than one game on any given day. For each team t and day d, the sum of all games involving team t on day d must be ≤ 1.

**Mathematical formulation**: ∑<sub>g: t plays in g</sub> x[g,d] ≤ 1 for all t ∈ T, d ∈ D

This prevents scheduling conflicts where a team would need to be in two places at once.


In [ ]:
#(b) Each team plays at most one game per day

In [ ]:
for t in T:
    for d in D:
        m.addConstr(
            quicksum(x[g, d] for g in G if home[g] == t or away[g] == t) <= 1,
            name=f"one_game_per_day_{t}_{d}"
        )

### Cell 13: Constraint (c) - Define Helper Variables
Defines two important helper variables:

1. **plays[t,d]**: Set equal to 1 if team t plays any game on day d
   - plays[t,d] = ∑<sub>g: t plays in g</sub> x[g,d]
   - This tracks whether a team is active on a given day

2. **tz_var[t,d]**: Set equal to the timezone offset of team t's game on day d
   - tz_var[t,d] = ∑<sub>g: t plays in g</sub> tz_game[g] × x[g,d]
   - If team t plays game g on day d, then tz_var[t,d] = tz_game[g] (the arena's UTC offset)
   - If team t doesn't play on day d, tz_var[t,d] = 0

These variables are essential for enforcing the timezone constraint.


In [ ]:
#(c) Define plays[t,d] and tz_var[t,d]

In [ ]:
for t in T:
    for d in D:
        # plays[t,d] = 1 if t plays any game g on day d
        m.addConstr(
            plays[t, d] == quicksum(x[g, d] for g in G if home[g] == t or away[g] == t),
            name=f"define_plays_{t}_{d}"
        )

        # tz_var[t,d] = time zone of arena of t's game that day
        # CRITICAL: Use tz_game_date[g, d] - offset depends on SCHEDULED date, not original date
        # This ensures DST is handled correctly when games are rescheduled
        m.addConstr(
            tz_var[t, d] == quicksum(
                tz_game_date[g, d] * x[g, d] for g in G if home[g] == t or away[g] == t
            ),
            name=f"define_tz_{t}_{d}"
        )

### Cell 13b: Part 2 Constraints - Preserve Original Home/Away Dates

This cell enforces Part 2 constraints (e) and (f):

- **Constraint (e)**: Each team must play home on the same dates as the original schedule
- **Constraint (f)**: Each team must play away on the same dates as the original schedule

**Mathematical formulation:**

For constraint (e) - Home dates:
- For each team t and each original home date d: ∑<sub>g: home[g] = t</sub> x[g, date_to_idx[d]] = 1
- This ensures team t plays exactly one home game on date d

For constraint (f) - Away dates:
- For each team t and each original away date d: ∑<sub>g: away[g] = t</sub> x[g, date_to_idx[d]] = 1
- This ensures team t plays exactly one away game on date d

These constraints preserve the original schedule's home/away date structure while allowing games to be rescheduled to different dates (subject to the timezone constraint).


In [ ]:
# Part 2 Constraints (e) and (f): Preserve original home/away dates

from collections import defaultdict

# Extract original home and away dates for each team
original_home_dates = defaultdict(set)
original_away_dates = defaultdict(set)

for _, row in games.iterrows():
    date_str = row['Date']
    home_team = row['Home']
    visitor_team = row['Visitor']

    # Store original home dates
    original_home_dates[home_team].add(date_str)

    # Store original away dates
    original_away_dates[visitor_team].add(date_str)

# Constraint (e): Each team plays home on the same dates as original
for team in T:
    for orig_date in original_home_dates.get(team, set()):
        if orig_date in date_to_idx:
            d = date_to_idx[orig_date]
            # Team must play exactly one home game on this date
            m.addConstr(
                quicksum(x[g, d] for g in G if home[g] == team) == 1,
                name=f"part2_home_date_{team}_{d}"
            )

# Constraint (f): Each team plays away on the same dates as original
for team in T:
    for orig_date in original_away_dates.get(team, set()):
        if orig_date in date_to_idx:
            d = date_to_idx[orig_date]
            # Team must play exactly one away game on this date
            m.addConstr(
                quicksum(x[g, d] for g in G if away[g] == team) == 1,
                name=f"part2_away_date_{team}_{d}"
            )

print(f"Added Part 2 constraints:")
print(f"  - Constraint (e): {sum(len(original_home_dates[t]) for t in T)} home date constraints")
print(f"  - Constraint (f): {sum(len(original_away_dates[t]) for t in T)} away date constraints")


Added Part 2 constraints:
  - Constraint (e): 128 home date constraints
  - Constraint (f): 128 away date constraints


### Cell 14: The Timezone Constraint (Core of Part 3)
This is the most important constraint for Part 3. It enforces that **no team plays three consecutive games where the sum of absolute timezone differences is ≥ 4**.

**The constraint logic:**
For each team t and each triple of consecutive days (d, d+1, d+2):

1. **Calculate differences:**
   - diff1[t,d] = tz_var[t,d+1] - tz_var[t,d] (timezone jump from day d to d+1)
   - diff2[t,d] = tz_var[t,d+2] - tz_var[t,d+1] (timezone jump from day d+1 to d+2)

2. **Calculate absolute values:**
   - a1[t,d] ≥ diff1[t,d] and a1[t,d] ≥ -diff1[t,d] → a1[t,d] = |diff1[t,d]|
   - a2[t,d] ≥ diff2[t,d] and a2[t,d] ≥ -diff2[t,d] → a2[t,d] = |diff2[t,d]|

3. **Enforce the constraint:**
   - If team t plays on all three days (d, d+1, d+2), then: a1[t,d] + a2[t,d] ≤ 3
   - If team t doesn't play on all three days, the constraint is relaxed using Big-M method
   - The Big-M constraint: a1[t,d] + a2[t,d] ≤ 3 + M × (3 - (plays[t,d] + plays[t,d+1] + plays[t,d+2]))
   - When the team plays all three days: plays[t,d] + plays[t,d+1] + plays[t,d+2] = 3, so the constraint becomes a1 + a2 ≤ 3
   - When the team doesn't play all three days: the right-hand side becomes ≥ 3 + M, effectively disabling the constraint

**Example:** If a team plays in timezone -4 (Eastern) on day d, -7 (Pacific) on day d+1, and -4 (Eastern) on day d+2:
- diff1 = -7 - (-4) = -3, so a1 = 3
- diff2 = -4 - (-7) = 3, so a2 = 3
- a1 + a2 = 6 ≥ 4, so this violates the constraint and won't be allowed


In [ ]:
M = 20  # Big-M, larger than any possible sum of time zone jumps

for t in T:
    for d in range(len(D) - 2):  # we look at triples (d, d+1, d+2)
        d1 = d + 1
        d2 = d + 2

        # diff1 = tz[t,d1] - tz[t,d]
        m.addConstr(
            diff1[t, d] == tz_var[t, d1] - tz_var[t, d],
            name=f"def_diff1_{t}_{d}"
        )

        # diff2 = tz[t,d2] - tz[t,d1]
        m.addConstr(
            diff2[t, d] == tz_var[t, d2] - tz_var[t, d1],
            name=f"def_diff2_{t}_{d}"
        )

        # a1 >= |diff1|
        m.addConstr(a1[t, d] >= diff1[t, d],   name=f"a1_pos_{t}_{d}")
        m.addConstr(a1[t, d] >= -diff1[t, d],  name=f"a1_neg_{t}_{d}")

        # a2 >= |diff2|
        m.addConstr(a2[t, d] >= diff2[t, d],   name=f"a2_pos_{t}_{d}")
        m.addConstr(a2[t, d] >= -diff2[t, d],  name=f"a2_neg_{t}_{d}")

        # Enforce sum of two jumps <= 3 *if* team plays in all three days
        m.addConstr(
            a1[t, d] + a2[t, d]
            <= 3 + M * (3 - (plays[t, d] + plays[t, d1] + plays[t, d2])),
            name=f"timezone_triple_{t}_{d}"
        )

In [ ]:
# Test different threshold values to find minimum feasible threshold
# This cell iteratively tests threshold values from 4 to 10

import time
from collections import defaultdict

print("="*80)
print("TESTING DIFFERENT TIMEZONE CONSTRAINT THRESHOLDS")
print("="*80)
print("Testing thresholds from 4 to 10 to find minimum feasible value")
print("All Part 2 constraints (e) and (f) will be preserved in each test")
print("="*80)

thresholds_to_test = range(4, 11)  # 4, 5, 6, 7, 8, 9, 10
min_feasible_threshold = None
results = {}

for threshold in thresholds_to_test:
    print(f"\n{'='*80}")
    print(f"Testing threshold = {threshold} (constraint: a1 + a2 <= {threshold})")
    print(f"{'='*80}")
    start_time = time.time()

    # Create new model for this threshold
    m_test = Model(f"NBA_TimeZone_Threshold_{threshold}")

    # Add all decision variables
    x_test = m_test.addVars(G, D, vtype=GRB.BINARY, name="x")
    plays_test = m_test.addVars(T, D, vtype=GRB.BINARY, name="plays")
    tz_var_test = m_test.addVars(T, D, lb=-12, ub=12, vtype=GRB.CONTINUOUS, name="tz")
    a1_test = m_test.addVars(T, D, lb=0.0, vtype=GRB.CONTINUOUS, name="a1")
    a2_test = m_test.addVars(T, D, lb=0.0, vtype=GRB.CONTINUOUS, name="a2")
    diff1_test = m_test.addVars(T, D, lb=-24, ub=24, vtype=GRB.CONTINUOUS, name="diff1")
    diff2_test = m_test.addVars(T, D, lb=-24, ub=24, vtype=GRB.CONTINUOUS, name="diff2")

    # Constraint (a): Each game scheduled exactly once
    for g in G:
        m_test.addConstr(quicksum(x_test[g, d] for d in D) == 1, name=f"one_day_per_game_{g}")

    # Constraint (b): Each team plays at most one game per day
    for t in T:
        for d in D:
            m_test.addConstr(
                quicksum(x_test[g, d] for g in G if home[g] == t or away[g] == t) <= 1,
                name=f"one_game_per_day_{t}_{d}"
            )

    # Constraint (c): Define helper variables
    for t in T:
        for d in D:
            # plays[t,d] = 1 if t plays any game g on day d
            m_test.addConstr(
                plays_test[t, d] == quicksum(x_test[g, d] for g in G if home[g] == t or away[g] == t),
                name=f"define_plays_{t}_{d}"
            )
            # tz_var[t,d] = time zone of arena of t's game that day
            m_test.addConstr(
                tz_var_test[t, d] == quicksum(
                    tz_game_date[g, d] * x_test[g, d] for g in G if home[g] == t or away[g] == t
                ),
                name=f"define_tz_{t}_{d}"
            )

    # Part 2 Constraints (e) and (f): Preserve original home/away dates
    for team in T:
        for orig_date in original_home_dates.get(team, set()):
            if orig_date in date_to_idx:
                d = date_to_idx[orig_date]
                m_test.addConstr(
                    quicksum(x_test[g, d] for g in G if home[g] == team) == 1,
                    name=f"part2_home_date_{team}_{d}"
                )

    for team in T:
        for orig_date in original_away_dates.get(team, set()):
            if orig_date in date_to_idx:
                d = date_to_idx[orig_date]
                m_test.addConstr(
                    quicksum(x_test[g, d] for g in G if away[g] == team) == 1,
                    name=f"part2_away_date_{team}_{d}"
                )

    # Timezone constraint with current threshold
    M = 20  # Big-M, larger than any possible sum of time zone jumps

    for t in T:
        for d in range(len(D) - 2):  # we look at triples (d, d+1, d+2)
            d1 = d + 1
            d2 = d + 2

            # diff1 = tz[t,d1] - tz[t,d]
            m_test.addConstr(
                diff1_test[t, d] == tz_var_test[t, d1] - tz_var_test[t, d],
                name=f"def_diff1_{t}_{d}"
            )

            # diff2 = tz[t,d2] - tz[t,d1]
            m_test.addConstr(
                diff2_test[t, d] == tz_var_test[t, d2] - tz_var_test[t, d1],
                name=f"def_diff2_{t}_{d}"
            )

            # a1 >= |diff1|
            m_test.addConstr(a1_test[t, d] >= diff1_test[t, d],   name=f"a1_pos_{t}_{d}")
            m_test.addConstr(a1_test[t, d] >= -diff1_test[t, d],  name=f"a1_neg_{t}_{d}")

            # a2 >= |diff2|
            m_test.addConstr(a2_test[t, d] >= diff2_test[t, d],   name=f"a2_pos_{t}_{d}")
            m_test.addConstr(a2_test[t, d] >= -diff2_test[t, d],  name=f"a2_neg_{t}_{d}")

            # Enforce sum of two jumps <= threshold *if* team plays in all three days
            m_test.addConstr(
                a1_test[t, d] + a2_test[t, d]
                <= threshold + M * (3 - (plays_test[t, d] + plays_test[t, d1] + plays_test[t, d2])),
                name=f"timezone_triple_{t}_{d}"
            )

    # Set objective and solve
    m_test.setObjective(0, GRB.MINIMIZE)
    m_test.Params.OutputFlag = 0  # Suppress solver output for cleaner results
    m_test.optimize()

    elapsed_time = time.time() - start_time
    results[threshold] = {
        'status': m_test.Status,
        'feasible': m_test.Status == GRB.OPTIMAL,
        'time': elapsed_time
    }

    if m_test.Status == GRB.OPTIMAL:
        print(f" FEASIBLE with threshold = {threshold}")
        print(f"   Solution found in {elapsed_time:.2f} seconds")
        min_feasible_threshold = threshold

        # Extract and save solution
        idx_to_date = {i: d for d, i in date_to_idx.items()}
        schedule_rows = []

        for g in G:
            for d in D:
                if x_test[g, d].X > 0.5:   # game g scheduled on day d
                    scheduled_date_str = idx_to_date[d]
                    scheduled_date = parse_date_string(scheduled_date_str)

                    arena = games.loc[g, "Arena"]
                    arena_tz = arena_timezone[arena]

                    if scheduled_date:
                        scheduled_offset = tz_offset_hours(arena_tz, scheduled_date)
                    else:
                        scheduled_offset = games.loc[g, "ArenaUTCOffset"]

                    schedule_rows.append({
                        "GameIndex": g,
                        "DateSlot": d,
                        "DateStr": scheduled_date_str,
                        "Visitor": games.loc[g, "Visitor"],
                        "Home": games.loc[g, "Home"],
                        "Arena": arena,
                        "ArenaUTCOffset": scheduled_offset
                    })

        final_schedule = pd.DataFrame(schedule_rows)
        final_schedule = final_schedule.sort_values(["DateSlot", "GameIndex"])

        # Export to CSV
        output_path = f"final_schedule_threshold_{threshold}.csv"
        final_schedule.to_csv(output_path, index=False)
        print(f"   Schedule saved to: {output_path}")
        print(f"   Total games: {len(final_schedule)}")

        break  # Stop at first feasible solution
    else:
        print(f" INFEASIBLE with threshold = {threshold}")
        print(f"   Status: {m_test.Status} (3=INFEASIBLE)")
        print(f"   Time: {elapsed_time:.2f} seconds")

# Final summary
print(f"\n{'='*80}")
print("SUMMARY OF THRESHOLD TESTING")
print(f"{'='*80}")
print(f"\nTested thresholds: {list(thresholds_to_test)}")
print(f"\nResults:")
for thresh, result in results.items():
    status_str = " FEASIBLE" if result['feasible'] else " INFEASIBLE"
    print(f"  Threshold {thresh}: {status_str} ({result['time']:.2f}s)")

if min_feasible_threshold is not None:
    print(f"\n MINIMUM FEASIBLE THRESHOLD: {min_feasible_threshold}")
    print(f"   This means the constraint allows: a1 + a2 <= {min_feasible_threshold}")
    print(f"   (i.e., sum of absolute timezone differences must be < {min_feasible_threshold + 1})")
else:
    print(f"\n  No feasible solution found for thresholds 4-10")
    print(f"   You may need to test higher threshold values or relax other constraints")


TESTING DIFFERENT TIMEZONE CONSTRAINT THRESHOLDS
Testing thresholds from 4 to 10 to find minimum feasible value
All Part 2 constraints (e) and (f) will be preserved in each test

Testing threshold = 4 (constraint: a1 + a2 <= 4)


GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

### Cell 15: Solve the Model
- Sets a dummy objective function (minimize 0) since we only need to find a feasible solution, not an optimal one
- Enables solver output to see the optimization progress
- Calls `m.optimize()` to solve the integer programming problem
- Prints the model status (2 = OPTIMAL, 3 = INFEASIBLE, etc.)

The solver will search for a schedule that satisfies all constraints. If a feasible solution exists, it will be found. If not, the solver will report that the problem is infeasible.


In [ ]:
# Dummy objective: just find any feasible schedule
m.setObjective(0, GRB.MINIMIZE)

m.Params.OutputFlag = 1  # turn on solver output if you want to see progress
m.optimize()

print("Model status:", m.Status)

### Cell 16: Extract the Solution
After the solver finds a feasible solution, this cell:
- Creates a reverse mapping from date indices back to actual date strings
- Iterates through all games and days to find which games are scheduled on which days (where x[g,d].X > 0.5)
- Builds a schedule DataFrame with game information, including visitor, home team, and scheduled date
- Sorts the schedule by date slot and game index for easy reading


In [ ]:
# Check if model has been optimized and has a solution
if m.Status == GRB.OPTIMAL:
    print(" Model solved successfully! Extracting solution...")

    # Map date index back to actual date string
    idx_to_date = {i: d for d, i in date_to_idx.items()}

    schedule = []
    for g in G:
        for d in D:
            if x[g, d].X > 0.5:  # game g is on day d
                schedule.append({
                    "GameIndex": g,
                    "Visitor": games.loc[g, "Visitor"],
                    "Home": games.loc[g, "Home"],
                    "DateSlot": d,
                    "DateStr": idx_to_date[d]
                })

    schedule_df = pd.DataFrame(schedule).sort_values(["DateSlot", "GameIndex"])
    print(f"Extracted {len(schedule)} games from solution")
    schedule_df.head(128)
    schedule_df.head(-20)
elif m.Status == GRB.INFEASIBLE:
    print(" ERROR: Model is INFEASIBLE")
    print("The constraints cannot all be satisfied simultaneously.")
    print("\n ROOT CAUSE:")
    print("   The original schedule (games.csv) has 26 timezone constraint violations.")
    print("   Part 2 constraints (e) and (f) force teams to play on the same dates,")
    print("   which keeps those violations, making it impossible to satisfy the timezone constraint.")
    print("\n SOLUTION:")
    print("   1. Comment out or skip Cell 13b (Part 2 constraints e and f)")
    print("   2. Re-run the optimization (Cell 15)")
    print("   3. The model will then find a feasible solution that satisfies the timezone constraint")
    print("\n   Alternatively, run Cell 15b to diagnose the infeasibility in detail.")
elif m.Status == GRB.INF_OR_UNBD:
    print(" ERROR: Model is INF_OR_UNBD (infeasible or unbounded)")
elif m.Status == GRB.UNBOUNDED:
    print(" ERROR: Model is UNBOUNDED")
else:
    print(f"  Model status: {m.Status}")
    print("Model has not been optimized yet or optimization failed.")
    print("Please run m.optimize() first.")

### Cell 17: Export Final Schedule to CSV
Creates a comprehensive schedule DataFrame with all relevant information:
- Game index, date slot, and actual date string
- Visitor and home teams
- Arena name and UTC offset

Then exports the schedule to `final_schedule.csv` for easy sharing and verification. This file can be used to verify that the timezone constraints are satisfied and to compare with the original schedule.


In [ ]:
import pandas as pd

# Check if model has been optimized and has a solution
if m.Status == GRB.OPTIMAL:
    print(" Model solved successfully! Exporting solution to CSV...")

    # Convert day-index -> actual date string used in the CSV
    idx_to_date = {i: d for d, i in date_to_idx.items()}

    # Build schedule list
    schedule_rows = []

    for g in G:
        for d in D:
            if x[g, d].X > 0.5:   # game g scheduled on day d
                # Get the scheduled date (not the original game date)
                scheduled_date_str = idx_to_date[d]
                scheduled_date = parse_date_string(scheduled_date_str)

                # Get arena and recalculate offset based on SCHEDULED date (not original date)
                arena = games.loc[g, "Arena"]
                arena_tz = arena_timezone[arena]

                # Calculate offset for the SCHEDULED date (handles DST correctly)
                if scheduled_date:
                    scheduled_offset = tz_offset_hours(arena_tz, scheduled_date)
                else:
                    scheduled_offset = games.loc[g, "ArenaUTCOffset"]  # fallback

                schedule_rows.append({
                    "GameIndex": g,
                    "DateSlot": d,
                    "DateStr": scheduled_date_str,
                    "Visitor": games.loc[g, "Visitor"],
                    "Home": games.loc[g, "Home"],
                    "Arena": arena,
                    "ArenaUTCOffset": scheduled_offset
                })

    # Convert to DataFrame
    final_schedule = pd.DataFrame(schedule_rows)

    # Sort nicely: by DateSlot then GameIndex
    final_schedule = final_schedule.sort_values(["DateSlot", "GameIndex"])

    # Export to CSV
    output_path = "final_schedule.csv"
    final_schedule.to_csv(output_path, index=False)

    print(f"Final schedule saved as: {output_path}")
    print(f"Total games in schedule: {len(final_schedule)}")
    final_schedule.head()
elif m.Status == GRB.INFEASIBLE:
    print(" ERROR: Model is INFEASIBLE - cannot export solution")
    print("The constraints cannot all be satisfied simultaneously.")
    print("\n ROOT CAUSE:")
    print("   The original schedule (games.csv) has 26 timezone constraint violations.")
    print("   Part 2 constraints (e) and (f) force teams to play on the same dates,")
    print("   which keeps those violations, making it impossible to satisfy the timezone constraint.")
    print("\n SOLUTION:")
    print("   1. Comment out or skip Cell 13b (Part 2 constraints e and f)")
    print("   2. Re-run the optimization (Cell 15)")
    print("   3. The model will then find a feasible solution that satisfies the timezone constraint")
    print("\n   Alternatively, run Cell 15b to diagnose the infeasibility in detail.")
else:
    print(f"  Model status: {m.Status}")
    print("Model has not been optimized yet or optimization failed.")
    print("Please run m.optimize() first and ensure it returns GRB.OPTIMAL.")

In [ ]:
# Print final schedule if it exists
if 'final_schedule' in globals():
    print(final_schedule)
else:
    print("final_schedule not available. Please run Cell 61 first to generate the schedule.")

In [ ]:
### Cell 18: Validation Report

This cell validates the final schedule against all Part 3 constraints and Part 2 constraints:

**Part 3 Constraints:**
1. **Chronological DateSlot Ordering**: Verifies DateSlots are in chronological order
2. **DST Handling**: Verifies timezone offsets are correct for each game's scheduled date
3. **Basic Constraints**: Verifies all games scheduled, no conflicts
4. **Timezone Constraint**: Verifies no team plays three consecutive games where sum of absolute timezone differences >= 4

**Part 2 Constraints:**
5. **Constraint (e)**: Each team plays home on the same dates as original schedule
6. **Constraint (f)**: Each team plays away on the same dates as original schedule
7. **Constraint (g)**: Each team plays home against each opponent the same number of times
8. **Constraint (h)**: Each team plays away against each opponent the same number of times

The validation will display detailed results for each check.


In [ ]:
# Comprehensive Validation of Final Schedule
import pandas as pd
from datetime import datetime
import pytz

def parse_date_string(date_str):
    """Parse date string in format 'Day, Mon DD, YYYY' to datetime object"""
    date_str = date_str.strip('"')
    try:
        return datetime.strptime(date_str, "%a, %b %d, %Y")
    except:
        return None

def tz_offset_hours(tz_name, ref_datetime=None):
    """Return integer UTC offset (in hours) for a given timezone."""
    if ref_datetime is None:
        ref_datetime = datetime.now()
    tz = pytz.timezone(tz_name)
    offset = tz.utcoffset(ref_datetime).total_seconds() / 3600
    return int(offset)

# Load schedules
schedule = pd.read_csv("final_schedule.csv")
games = pd.read_csv("games.csv")

print("=" * 80)
print("COMPREHENSIVE VALIDATION FOR PART 3 - NBA SCHEDULE")
print("=" * 80)


print("\n" + "=" * 80)
print("VALIDATION 1: Chronological DateSlot Ordering")
print("=" * 80)

date_slots = schedule[['DateSlot', 'DateStr']].drop_duplicates().sort_values('DateSlot')
chronological_errors = []
for i in range(len(date_slots) - 1):
    curr_date = parse_date_string(date_slots.iloc[i]['DateStr'])
    next_date = parse_date_string(date_slots.iloc[i+1]['DateStr'])
    if curr_date and next_date and curr_date >= next_date:
        chronological_errors.append({
            'DateSlot': date_slots.iloc[i]['DateSlot'],
            'Date': date_slots.iloc[i]['DateStr'],
            'NextDateSlot': date_slots.iloc[i+1]['DateSlot'],
            'NextDate': date_slots.iloc[i+1]['DateStr']
        })

if chronological_errors:
    print(f"    ERROR: Found {len(chronological_errors)} chronological ordering issues")
else:
    print("    PASS: All DateSlots are in chronological order")
    print("\n   DateSlot to Date mapping:")
    for _, row in date_slots.iterrows():
        print(f"      DateSlot {int(row['DateSlot']):2d}: {row['DateStr']}")


print("\n" + "=" * 80)
print("VALIDATION 2: DST Handling - Correct Timezone Offsets")
print("=" * 80)

dst_errors = []
for idx, row in schedule.iterrows():
    game_date = parse_date_string(row['DateStr'])
    if not game_date:
        continue
    arena = row['Arena']
    expected_tz = arena_timezone.get(arena)
    if not expected_tz:
        continue
    expected_offset = tz_offset_hours(expected_tz, game_date)
    actual_offset = int(row['ArenaUTCOffset'])
    if expected_offset != actual_offset:
        dst_errors.append({
            'GameIndex': row['GameIndex'],
            'Date': row['DateStr'],
            'Arena': arena,
            'Expected': expected_offset,
            'Actual': actual_offset
        })

if dst_errors:
    print(f"    ERROR: Found {len(dst_errors)} DST offset mismatches")
    for err in dst_errors[:5]:
        print(f"      Game {err['GameIndex']}: {err['Date']} at {err['Arena']}")
        print(f"         Expected: {err['Expected']}, Actual: {err['Actual']}")
else:
    print("    PASS: All timezone offsets are correct for their dates")


print("\n" + "=" * 80)
print("VALIDATION 3: Basic Constraints")
print("=" * 80)

game_counts = schedule['GameIndex'].value_counts()
duplicate_games = game_counts[game_counts > 1]
missing_games = set(games.index) - set(schedule['GameIndex'])

if len(duplicate_games) > 0:
    print(f"    ERROR: {len(duplicate_games)} games scheduled multiple times")
else:
    print("    PASS: All games scheduled exactly once")

if len(missing_games) > 0:
    print(f"    ERROR: {len(missing_games)} games missing from schedule")
else:
    print("    PASS: All games present in schedule")

team_date_violations = []
for team in set(schedule['Home']).union(schedule['Visitor']):
    team_games = schedule[(schedule['Home'] == team) | (schedule['Visitor'] == team)]
    date_slot_counts = team_games['DateSlot'].value_counts()
    duplicates = date_slot_counts[date_slot_counts > 1]
    if len(duplicates) > 0:
        team_date_violations.append({'Team': team, 'DateSlots': list(duplicates.index)})

if team_date_violations:
    print(f"    ERROR: {len(team_date_violations)} teams play multiple games on same DateSlot")
else:
    print("    PASS: No team plays twice on same DateSlot")


print("\n" + "=" * 80)
print("VALIDATION 4: Timezone Constraint (Part 3 Core Requirement)")
print("=" * 80)
print("\n   Constraint: No team should play three consecutive games (by DateSlot)")
print("   where sum of absolute timezone differences >= 4")
print("   Formula: |tz[d+1] - tz[d]| + |tz[d+2] - tz[d+1]| < 4")

all_teams = sorted(set(schedule['Home']).union(schedule['Visitor']))
violations = []
total_triples_checked = 0

for team in all_teams:
    team_games = schedule[(schedule['Home'] == team) | (schedule['Visitor'] == team)].copy()
    team_games = team_games.sort_values('DateSlot').reset_index(drop=True)

    if len(team_games) < 3:
        continue

    for i in range(len(team_games) - 2):
        game1 = team_games.iloc[i]
        game2 = team_games.iloc[i+1]
        game3 = team_games.iloc[i+2]

        slot1 = int(game1['DateSlot'])
        slot2 = int(game2['DateSlot'])
        slot3 = int(game3['DateSlot'])

        if slot2 == slot1 + 1 and slot3 == slot2 + 1:
            total_triples_checked += 1
            tz1 = int(game1['ArenaUTCOffset'])
            tz2 = int(game2['ArenaUTCOffset'])
            tz3 = int(game3['ArenaUTCOffset'])

            diff1 = abs(tz2 - tz1)
            diff2 = abs(tz3 - tz2)
            total_diff = diff1 + diff2

            if total_diff >= 4:
                violations.append({
                    'Team': team,
                    'DateSlot1': slot1, 'Date1': game1['DateStr'], 'TZ1': tz1,
                    'DateSlot2': slot2, 'Date2': game2['DateStr'], 'TZ2': tz2,
                    'DateSlot3': slot3, 'Date3': game3['DateStr'], 'TZ3': tz3,
                    'Diff1': diff1, 'Diff2': diff2, 'TotalDiff': total_diff
                })

print(f"\n   Checked {total_triples_checked} consecutive DateSlot triples across all teams")

if violations:
    print(f"\n    ERROR: Found {len(violations)} timezone constraint violations:")
    for v in violations[:5]:
        print(f"\n      Team: {v['Team']}")
        print(f"         DateSlot {v['DateSlot1']} ({v['Date1']}): TZ={v['TZ1']}")
        print(f"         DateSlot {v['DateSlot2']} ({v['Date2']}): TZ={v['TZ2']}")
        print(f"         DateSlot {v['DateSlot3']} ({v['Date3']}): TZ={v['TZ3']}")
        print(f"         |{v['TZ2']} - {v['TZ1']}| + |{v['TZ3']} - {v['TZ2']}| = {v['Diff1']} + {v['Diff2']} = {v['TotalDiff']} >= 4 ❌")
else:
    print("\n    PASS: All consecutive DateSlot triples satisfy timezone constraint")


print("\n" + "=" * 80)
print("VALIDATION 5: Part 2 Constraints")
print("=" * 80)
print("\n   Part 2 requires:")
print("   (e) Each team plays home on the same dates as original")
print("   (f) Each team plays away on the same dates as original")
print("   (g) Each team plays home against each opponent the same number of times")
print("   (h) Each team plays away against each opponent the same number of times")

from collections import defaultdict

# Load original games
original_games = pd.read_csv("games.csv")

# Extract original home and away dates
original_home_dates = defaultdict(set)
original_away_dates = defaultdict(set)
for _, row in original_games.iterrows():
    date = row['Date']
    original_home_dates[row['Home']].add(date)
    original_away_dates[row['Visitor']].add(date)

# Extract final home and away dates
final_home_dates = defaultdict(set)
final_away_dates = defaultdict(set)
for _, row in schedule.iterrows():
    date = row['DateStr']
    final_home_dates[row['Home']].add(date)
    final_away_dates[row['Visitor']].add(date)

# Check constraint (e): Home dates preserved
home_date_violations = []
for team in sorted(set(original_home_dates.keys()) | set(final_home_dates.keys())):
    orig_dates = original_home_dates.get(team, set())
    final_dates = final_home_dates.get(team, set())
    if orig_dates != final_dates:
        home_date_violations.append({
            'team': team,
            'missing': len(orig_dates - final_dates),
            'extra': len(final_dates - orig_dates)
        })

# Check constraint (f): Away dates preserved
away_date_violations = []
for team in sorted(set(original_away_dates.keys()) | set(final_away_dates.keys())):
    orig_dates = original_away_dates.get(team, set())
    final_dates = final_away_dates.get(team, set())
    if orig_dates != final_dates:
        away_date_violations.append({
            'team': team,
            'missing': len(orig_dates - final_dates),
            'extra': len(final_dates - orig_dates)
        })

# Check constraint (g): Home matchup counts preserved
original_home_matchups = defaultdict(lambda: defaultdict(int))
for _, row in original_games.iterrows():
    original_home_matchups[row['Home']][row['Visitor']] += 1

final_home_matchups = defaultdict(lambda: defaultdict(int))
for _, row in schedule.iterrows():
    final_home_matchups[row['Home']][row['Visitor']] += 1

home_matchup_violations = []
for team in sorted(set(original_home_matchups.keys()) | set(final_home_matchups.keys())):
    orig = original_home_matchups.get(team, {})
    final = final_home_matchups.get(team, {})
    all_opponents = set(orig.keys()) | set(final.keys())
    for opponent in all_opponents:
        if orig.get(opponent, 0) != final.get(opponent, 0):
            home_matchup_violations.append({'team': team, 'opponent': opponent})

# Check constraint (h): Away matchup counts preserved
original_away_matchups = defaultdict(lambda: defaultdict(int))
for _, row in original_games.iterrows():
    original_away_matchups[row['Visitor']][row['Home']] += 1

final_away_matchups = defaultdict(lambda: defaultdict(int))
for _, row in schedule.iterrows():
    final_away_matchups[row['Visitor']][row['Home']] += 1

away_matchup_violations = []
for team in sorted(set(original_away_matchups.keys()) | set(final_away_matchups.keys())):
    orig = original_away_matchups.get(team, {})
    final = final_away_matchups.get(team, {})
    all_opponents = set(orig.keys()) | set(final.keys())
    for opponent in all_opponents:
        if orig.get(opponent, 0) != final.get(opponent, 0):
            away_matchup_violations.append({'team': team, 'opponent': opponent})

# Report results
print("\n   Constraint (e) - Home dates preserved:")
if len(home_date_violations) == 0:
    print("       PASS: All teams play home on same dates as original")
else:
    print(f"       FAIL: {len(home_date_violations)} teams have different home dates")
    for v in home_date_violations[:3]:
        print(f"         {v['team']}: {v['missing']} missing, {v['extra']} extra dates")

print("\n   Constraint (f) - Away dates preserved:")
if len(away_date_violations) == 0:
    print("       PASS: All teams play away on same dates as original")
else:
    print(f"       FAIL: {len(away_date_violations)} teams have different away dates")
    for v in away_date_violations[:3]:
        print(f"         {v['team']}: {v['missing']} missing, {v['extra']} extra dates")

print("\n   Constraint (g) - Home matchup counts preserved:")
if len(home_matchup_violations) == 0:
    print("       PASS: All home matchup counts match original")
else:
    print(f"       FAIL: {len(home_matchup_violations)} home matchup count violations")

print("\n   Constraint (h) - Away matchup counts preserved:")
if len(away_matchup_violations) == 0:
    print("       PASS: All away matchup counts match original")
else:
    print(f"       FAIL: {len(away_matchup_violations)} away matchup count violations")

print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

all_passed = (
    len(chronological_errors) == 0 and
    len(dst_errors) == 0 and
    len(duplicate_games) == 0 and
    len(missing_games) == 0 and
    len(team_date_violations) == 0 and
    len(violations) == 0 and
    len(home_date_violations) == 0 and
    len(away_date_violations) == 0 and
    len(home_matchup_violations) == 0 and
    len(away_matchup_violations) == 0
)

if all_passed:
    print("\n    ALL VALIDATIONS PASSED!")
    print("\n   The schedule correctly:")
    print("      - Orders DateSlots chronologically")
    print("      - Uses correct DST offsets for each game date")
    print("      - Schedules all games exactly once")
    print("      - Prevents teams from playing twice on same DateSlot")
    print("      - Satisfies timezone constraint for all consecutive DateSlot triples")
    print("      - Preserves original home dates (Part 2 constraint e)")
    print("      - Preserves original away dates (Part 2 constraint f)")
    print("      - Preserves home matchup counts (Part 2 constraint g)")
    print("      - Preserves away matchup counts (Part 2 constraint h)")
    print(f"\n   Statistics:")
    print(f"      - Total games: {len(schedule)}")
    print(f"      - DateSlots: {len(date_slots)}")
    print(f"      - Teams: {len(all_teams)}")
    print(f"      - Consecutive triples checked: {total_triples_checked}")
    print(f"      - Constraint violations: 0")
else:
    print("\n    SOME VALIDATIONS FAILED")
    print("\n   Part 3 Issues:")
    if len(chronological_errors) > 0:
        print(f"      - {len(chronological_errors)} chronological ordering errors")
    if len(dst_errors) > 0:
        print(f"      - {len(dst_errors)} DST offset errors")
    if len(duplicate_games) > 0:
        print(f"      - {len(duplicate_games)} duplicate game schedules")
    if len(missing_games) > 0:
        print(f"      - {len(missing_games)} missing games")
    if len(team_date_violations) > 0:
        print(f"      - {len(team_date_violations)} teams playing twice on same DateSlot")
    if len(violations) > 0:
        print(f"      - {len(violations)} timezone constraint violations")
    print("\n   Part 2 Constraint Issues:")
    if len(home_date_violations) > 0:
        print(f"      - Constraint (e): {len(home_date_violations)} teams have different home dates")
    if len(away_date_violations) > 0:
        print(f"      - Constraint (f): {len(away_date_violations)} teams have different away dates")
    if len(home_matchup_violations) > 0:
        print(f"      - Constraint (g): {len(home_matchup_violations)} home matchup count violations")
    if len(away_matchup_violations) > 0:
        print(f"      - Constraint (h): {len(away_matchup_violations)} away matchup count violations")

    if len(home_date_violations) > 0 or len(away_date_violations) > 0:
        print("\n     RECOMMENDATION:")
        print("      To satisfy Part 2 constraints (e) and (f), add constraints to the model")
        print("      that enforce each team plays home/away on the same dates as the original schedule.")

print("\n" + "=" * 80)


### Part 4: Distance-Fair Schedule Optimization

This cell builds an Integer Programming model in **PuLP** to redesign the schedule:

- **Preserves original home/away dates:** for every team we keep exactly the same home and away dates as in the original schedule.
- **Allows new opponents:** opponents can change on those dates so the solver can reduce total travel distances.
- **Uses geographic coordinates:** team latitude/longitude and the **haversine formula** are used to compute distances (km) between cities.
- **Defines decision variables:** y[i,j,d] (home i vs away j on date d), total away-travel distance Dist[t] for each team, and the mean distance mu.
- **Models fairness with L1 deviations:** variables dev_pos[t] and dev_neg[t] capture |Dist[t] − mu| so that uneven travel can be penalized in the objective.

The objective minimizes the **mean distance** mu plus a small **fairness term** based on L1 deviation, so the new schedule both reduces overall travel and spreads it more evenly across teams. Finally, the cell writes the optimized schedule to CSV and compares original vs new distances and L1 variance to evaluate the gain in fairness.


In [64]:
import pandas as pd
import math
from google.colab import drive
import pulp as pl


drive.mount('/content/drive')

CSV_PATH = "/content/drive/MyDrive/games.csv"
games = pd.read_csv(CSV_PATH)



teams = sorted(set(games["Home"]).union(games["Visitor"]))
T = teams

dates = sorted(games["Date"].unique())
D = list(range(len(dates)))               # indices 0..|D|-1
date_to_idx = {d: i for i, d in enumerate(dates)}

# Home/away dates per team in the ORIGINAL schedule
home_dates = {t: set() for t in T}
away_dates = {t: set() for t in T}

for _, row in games.iterrows():
    d = row["Date"]
    h = row["Home"]
    a = row["Visitor"]
    home_dates[h].add(d)
    away_dates[a].add(d)

# Map each team to its home arena (for printing schedule)
team_arena = {}
for _, row in games.iterrows():
    h = row["Home"]
    if h not in team_arena:
        team_arena[h] = row["Arena"]

# Coordinates (lat, lon) for each TEAM Stadium
team_coords = {
    "Atlanta Hawks": (33.954880, -84.295410),
    "Boston Celtics": (42.366581, -71.061630),
    "Chicago Bulls": (41.880650, -87.675140),
    "Cleveland Cavaliers": (41.496891, -81.689636),
    "Denver Nuggets": (39.747921, -105.006798),
    "Dallas Mavericks": (32.7905076, -96.8102721),
    "Golden State Warriors": (37.7679001, -122.3874223),
    "Houston Rockets": (29.7507473, -95.3622315),
    "Philadelphia 76ers": (39.9011004, -75.1720165),
    "Los Angeles Lakers": (34.0414696, -118.267296),
    "Miami Heat": (25.7813595, -80.1879435),
    "Milwaukee Bucks": (43.0450096, -87.9174871),
    "Toronto Raptors": (43.6434338, -79.3790777),
    "Phoenix Suns": (33.4465114, -112.0706267),
    "Brooklyn Nets": (40.6825106, -73.9752519),
    "New York Knicks": (40.7505129, -73.9935159),
}

def haversine(lat1, lon1, lat2, lon2):
    """Compute great-circle distance (km) between two points."""
    R = 6371.0  # Earth radius in km
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    return 2 * R * math.asin(math.sqrt(a))

# Pre-compute distance(home, away) = distance from away city to home city
dist_team = {}
for i in T:          # home team
    for j in T:      # away team
        if i == j:
            continue
        if i in team_coords and j in team_coords:
            lat_h, lon_h = team_coords[i]   # home team location
            lat_a, lon_a = team_coords[j]   # away team location
            d_ij = haversine(lat_a, lon_a, lat_h, lon_h)
        else:
            d_ij = 0.0
        dist_team[(i, j)] = d_ij



orig_dist = {t: 0.0 for t in T}
for _, row in games.iterrows():
    home = row["Home"]
    away = row["Visitor"]
    orig_dist[away] += dist_team[(home, away)]

orig_mu = sum(orig_dist.values()) / len(T)
print("Original average distance (mu original):", orig_mu)



prob = pl.LpProblem("NBA_Distance_L1_with_mean", pl.LpMinimize)

# Binary decision variables: y[i,j,d] = 1 if on date d
# team i plays at home vs team j (away)
y = pl.LpVariable.dicts(
    "y",
    ((i, j, d_idx) for i in T for j in T for d_idx in D if i != j),
    lowBound=0,
    upBound=1,
    cat=pl.LpBinary,
)

# Total distance per team in the NEW schedule
Dist = pl.LpVariable.dicts("Dist", T, lowBound=0, cat=pl.LpContinuous)

# Mean distance in the NEW schedule
mu = pl.LpVariable("mu", lowBound=0, cat=pl.LpContinuous)

# Positive and negative deviation from the new mean mu
dev_pos = pl.LpVariable.dicts("dev_pos", T, lowBound=0, cat=pl.LpContinuous)
dev_neg = pl.LpVariable.dicts("dev_neg", T, lowBound=0, cat=pl.LpContinuous)


# Keep same home/away dates and number of games for each team.

for t in T:
    for d_str in dates:
        d_idx = date_to_idx[d_str]

        # Sum of games where t is home / away on date d
        home_games_td = pl.lpSum(
            y[(t, j, d_idx)] for j in T if j != t
        )
        away_games_td = pl.lpSum(
            y[(i, t, d_idx)] for i in T if i != t
        )

        is_home = d_str in home_dates[t]
        is_away = d_str in away_dates[t]

        if is_home and is_away:
            raise ValueError(f"Team {t} is both home and away on {d_str}")

        if is_home:
            # t must play exactly one home game and zero away games that day
            prob += (home_games_td == 1, f"home_day_{t}_{d_idx}")
            prob += (away_games_td == 0, f"no_away_if_home_{t}_{d_idx}")
        elif is_away:
            # t must play exactly one away game and zero home games that day
            prob += (away_games_td == 1, f"away_day_{t}_{d_idx}")
            prob += (home_games_td == 0, f"no_home_if_away_{t}_{d_idx}")
        else:
            # t does not play that day
            prob += (home_games_td == 0, f"no_home_{t}_{d_idx}")
            prob += (away_games_td == 0, f"no_away_{t}_{d_idx}")

# (IMPORTANT) We DO NOT keep the same pairs (i vs j),
# so the solver is free to re-match teams to improve distances.

# 7. Distance and mean definition

# Total travel distance for each team (as away team) in the NEW schedule
for t in T:
    prob += (
        Dist[t]
        == pl.lpSum(
            dist_team[(i, t)] * y[(i, t, d_idx)]
            for i in T if i != t
            for d_idx in D
        ),
        f"def_Dist_{t}",
    )

# Mean distance mu = (1/|T|) * sum_t Dist[t]
prob += (
    mu * len(T) == pl.lpSum(Dist[t] for t in T),
    "def_mu"
)

# Deviation from the new mean mu:
#   Dist[t] - mu = dev_pos[t] - dev_neg[t]
for t in T:
    prob += (
        Dist[t] - mu == dev_pos[t] - dev_neg[t],
        f"def_dev_{t}",
    )


#   minimize mu  +  alpha * sum_t |Dist[t] - mu|

alpha = 0.001   # small weight for fairness term; you can tune this
prob += mu + alpha *50* (pl.lpSum(dev_pos[t] + dev_neg[t] for t in T))



solver = pl.PULP_CBC_CMD(msg=True)
prob.solve(solver)

print("Status:", pl.LpStatus[prob.status])
print("New mean distance mu* =", pl.value(mu))


new_schedule_rows = []

for d_idx in D:
    d_str = dates[d_idx]
    for i in T:
        for j in T:
            if i == j:
                continue
            key = (i, j, d_idx)
            if key in y and pl.value(y[key]) > 0.5:
                new_schedule_rows.append({
                    "Date": d_str,
                    "Home": i,
                    "Visitor": j,
                    "Arena": team_arena[i],  # location of the game
                    "DistanceAwayTeam_km": dist_team[(i, j)]
                })

new_schedule = pd.DataFrame(new_schedule_rows)
new_schedule["Date"] = pd.to_datetime(new_schedule["Date"])
new_schedule = new_schedule.sort_values(["Date", "Home", "Visitor"]).reset_index(drop=True)

# Save sorted schedule to Google Drive
output_path_drive = "/content/drive/MyDrive/schedule_q4_distance_L1_newmean_sorted.csv"
new_schedule.to_csv(output_path_drive, index=False)
print("Sorted schedule saved to:", output_path_drive)



# New distances per team from the model
new_dist = {t: pl.value(Dist[t]) for t in T}
new_mu = pl.value(mu)

rows = []
for t in T:
    old_d = orig_dist[t]
    new_d = new_dist[t]
    diff_abs = abs(new_d - old_d)  # <- nessun valore negativo

    rows.append({
        "Team": t,
        "OrigDistance_km": old_d,
        "NewDistance_km": new_d,
        "DiffAbs_km": diff_abs
    })

dist_comp = pd.DataFrame(rows)
print("\n=== Distance comparison with absolute differences ===")
print(dist_comp)

# Mean absolute difference
mean_abs_diff = dist_comp["DiffAbs_km"].mean()
print("\nMean absolute distance difference (km):", mean_abs_diff)

# L1 "variance" = mean absolute deviation from the mean
orig_L1_var = sum(abs(orig_dist[t] - orig_mu) for t in T) / len(T)
new_L1_var  = sum(abs(new_dist[t]  - new_mu)  for t in T) / len(T)

print("\nOriginal L1 variance:", orig_L1_var)
print("New L1 variance:", new_L1_var)

print("\n=== Fairness Evaluation (L1 metric) ===")
if new_L1_var < orig_L1_var:
    print("The new schedule REDUCES the L1 variance → fairer distribution of travel distances.")
elif new_L1_var > orig_L1_var:
    print("The new schedule INCREASES the L1 variance → less fair distribution.")
else:
    print("The fairness level is unchanged (same L1 variance).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original average distance (mu original): 15417.935389937518
Status: Optimal
New mean distance mu* = 11680.664
Sorted schedule saved to: /content/drive/MyDrive/schedule_q4_distance_L1_newmean_sorted.csv

=== Distance comparison with absolute differences ===
                     Team  OrigDistance_km  NewDistance_km    DiffAbs_km
0           Atlanta Hawks      7113.702232       8381.7580   1268.055768
1          Boston Celtics     18964.259739      10618.3110   8345.948739
2           Brooklyn Nets     15840.409864      11230.1460   4610.263864
3           Chicago Bulls      9191.328592      11357.5180   2166.189408
4     Cleveland Cavaliers     12347.774023       6480.5440   5867.230023
5        Dallas Mavericks     12666.820983      10496.8600   2169.960983
6          Denver Nuggets     13229.781187      11817.8700   1411.911187
7   Golden State Warriors     